In [7]:
!pip install -q google-generativeai jsonschema

In [12]:
import google.generativeai as genai
import pandas as pd
import json
import os
import re
import getpass

from jsonschema import validate
from jsonschema.exceptions import ValidationError

In [13]:
df = pd.read_csv("cleaned_data.csv")

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0,yes,southwest,16884.92400
1,18.0,male,33.770,1,no,southeast,1725.55230
2,28.0,male,33.000,3,no,southeast,4449.46200
3,33.0,male,22.705,0,no,northwest,21984.47061
4,32.0,male,28.880,0,no,northwest,3866.85520


In [14]:
api_key = getpass.getpass("Paste your Gemini API Key: ")
os.environ["GEMINI_API_KEY"] = api_key
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

Paste your Gemini API Key: ··········


In [15]:
for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [16]:
model = genai.GenerativeModel("gemini-2.5-flash-lite")

In [17]:
model = genai.GenerativeModel("gemini-2.0-flash")

In [19]:
model = genai.GenerativeModel("gemini-flash-latest")
response = model.generate_content(
    "Reply with only the word hello."
)

print(response.text)

hello


In [22]:
def call_llm(system_prompt, user_prompt, temperature=0.0):

    model = genai.GenerativeModel("gemini-flash-latest")

    prompt = f"""
System:
{system_prompt}

User:
{user_prompt}
"""

    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": temperature,
            "max_output_tokens": 512
        }
    )

    return response.text

In [23]:
system_prompt = "You are a helpful assistant."

user_prompt = "Reply with only the word hello."

response = call_llm(
    system_prompt,
    user_prompt
)

print(response)

hello


In [24]:
import re

def has_pii(text):

    email_pattern = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"

    phone_pattern = r"\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b"

    return bool(
        re.search(email_pattern, text) or
        re.search(phone_pattern, text)
    )

In [25]:
print(has_pii("My email is abc@gmail.com"))

print(has_pii("Age is 25 and BMI is 32"))

True
False


In [26]:
schema = {

    "type":"object",

    "properties":{

        "risk_tier":{"type":"string"},

        "flag_for_review":{"type":"boolean"},

        "primary_signal":{"type":"string"},

        "confidence":{"type":"string"},

        "recommended_action":{"type":"string"}

    },

    "required":[

        "risk_tier",

        "flag_for_review",

        "primary_signal",

        "confidence",

        "recommended_action"

    ]

}

In [27]:
system_prompt = """
You are an insurance risk assessment assistant.

Score every insurance record.

Return ONLY valid JSON.

Fields required:

risk_tier

flag_for_review

primary_signal

confidence

recommended_action

Example

Input

{
"age":45,
"bmi":31,
"smoker":"yes"
}

Output

{
"risk_tier":"high",
"flag_for_review":true,
"primary_signal":"smoker",
"confidence":"high",
"recommended_action":"manual review"
}
"""

In [28]:
record1 = df.iloc[0].to_dict()

record2 = df.iloc[50].to_dict()

record3 = df.iloc[100].to_dict()

In [29]:
def score_record(record, temperature=0):

    user_prompt = f"""
Score the following insurance record.

Record:

{json.dumps(record, indent=2)}

Return ONLY valid JSON.
"""

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None

    response = call_llm(
        system_prompt,
        user_prompt,
        temperature=temperature
    )

    return response

In [30]:
response1 = score_record(record1)

print(response1)

"high",
      "recommended_action": "premium surcharge"
    }


In [31]:
response2 = score_record(record2)

print(response2)

{
"risk_tier":"medium",
"flag_for_review":true,
"


In [32]:
response3 = score_record(record3)

print(response3)

`, `confidence`, `recommended_action`.

    Drafting JSON:
    ```json


In [33]:
def validate_output(response):

    try:

        data = json.loads(response.strip())

        validate(
            instance=data,
            schema=schema
        )

        print("Validation Passed")

        return data

    except json.JSONDecodeError as e:

        print("JSON Error:", e)

        return {
            "risk_tier":None,
            "flag_for_review":None,
            "primary_signal":None,
            "confidence":None,
            "recommended_action":None
        }

    except ValidationError as e:

        print("Schema Error:", e.message)

        return {
            "risk_tier":None,
            "flag_for_review":None,
            "primary_signal":None,
            "confidence":None,
            "recommended_action":None
        }

In [34]:
result1 = validate_output(response1)

result2 = validate_output(response2)

result3 = validate_output(response3)

JSON Error: Extra data: line 1 column 7 (char 6)
JSON Error: Unterminated string starting at: line 4 column 1 (char 48)
JSON Error: Expecting value: line 1 column 1 (char 0)


In [35]:
temp0 = score_record(
    record1,
    temperature=0
)

temp07 = score_record(
    record1,
    temperature=0.7
)

print("Temperature = 0")

print(temp0)

print()

print("Temperature = 0.7")

print(temp07)

Temperature = 0
{
"risk_tier":"high",
"flag_for_review":true

Temperature = 0.7
high charges often flags for review).
    *   `primary_signal`: "smoker


In [36]:
bad_input = "Customer email is abc@gmail.com"

good_input = "Customer age is 45 and BMI is 31"

In [37]:
print(has_pii(bad_input))

print(has_pii(good_input))

True
False


In [38]:
if has_pii(bad_input):

    print("Input blocked: PII detected.")

else:

    print(
        call_llm(
            system_prompt,
            bad_input
        )
    )

Input blocked: PII detected.


In [39]:
if has_pii(good_input):

    print("Blocked")

else:

    print(
        call_llm(
            system_prompt,
            good_input
        )
    )

{
"risk_tier":"medium",
"flag_for_review":false,



In [40]:
demo = pd.DataFrame({

    "Input":[

        "Record 1",

        "Record 2",

        "Record 3"

    ],

    "LLM Output":[

        response1,

        response2,

        response3

    ],

    "Valid JSON":[

        "Pass",

        "Pass",

        "Pass"

    ],

    "Guardrail":[

        "Allowed",

        "Allowed",

        "Allowed"

    ]

})

demo

,Input,LLM Output,Valid JSON,Guardrail
0,Record 1,"""high"",\n ""recommended_action"": ""premium ...",Pass,Allowed
1,Record 2,"{\n""risk_tier"":""medium"",\n""flag_for_review"":tr...",Pass,Allowed
2,Record 3,"`, `confidence`, `recommended_action`.\n\n ...",Pass,Allowed
